In [1]:
import tensorflow as tf
import numpy as np
from pathlib import Path
from collections import Counter

SEED = 0

files = [
    "Kaggle_Data_recheck/Incorrect_Training_Data/training10_0.tfrecords",
    "Kaggle_Data_recheck/Incorrect_Training_Data/training10_1.tfrecords",
    "Kaggle_Data_recheck/Incorrect_Training_Data/training10_2.tfrecords",
    "Kaggle_Data_recheck/Incorrect_Training_Data/training10_3.tfrecords",
    "Kaggle_Data_recheck/Incorrect_Training_Data/training10_4.tfrecords",
]

raw = tf.data.TFRecordDataset(files)

for i, r in enumerate(raw.take(3)):
    ex = tf.train.Example.FromString(r.numpy())
    print(f"Example {i}:")
    for k, v in ex.features.feature.items():
        if v.int64_list.value:
            print(k, v.int64_list.value[:5])
        elif v.float_list.value:
            print(k, v.float_list.value[:5])
        elif v.bytes_list.value:
            print(k, f"bytes[{len(v.bytes_list.value[0])}]")


2026-01-28 11:53:01.933622: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-28 11:53:01.951160: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-01-28 11:53:01.972936: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-01-28 11:53:01.979932: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-28 11:53:01.995329: I tensorflow/core/platform/cpu_feature_guar

Example 0:
filename bytes[24]
label_normal [0]
image bytes[89401]
label [0]
Example 1:
filename bytes[24]
label_normal [0]
image bytes[89401]
label [0]
Example 2:
filename bytes[24]
label_normal [0]
image bytes[89401]
label [0]


I0000 00:00:1769601184.407083   23578 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769601184.469678   23578 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769601184.472265   23578 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1769601184.477115   23578 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

In [2]:
SRC = Path("Kaggle_Data_recheck/Incorrect_Training_Data")
DST = Path("Kaggle_Data_recheck/Cleaned_Training_Data")
DST.mkdir(exist_ok=True)

def remap(v):
    if v == 2:
        return 1
    if v >= 3:
        return 2
    return v

for tfrec in SRC.glob("training10_*.tfrecords"):
    out = DST / tfrec.name
    writer = tf.io.TFRecordWriter(str(out))

    for r in tf.data.TFRecordDataset(str(tfrec)):
        ex = tf.train.Example.FromString(r.numpy())

        # drop label_normal
        ex.features.feature.pop("label_normal", None)

        # remap label
        lbl = ex.features.feature["label"].int64_list.value[0]
        ex.features.feature["label"].int64_list.value[0] = remap(lbl)

        writer.write(ex.SerializeToString())

    writer.close()

2026-01-28 12:03:47.418563: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
2026-01-28 12:03:59.217977: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [3]:
labels = []
for r in tf.data.TFRecordDataset(str(out)).take(1000):
    ex = tf.train.Example.FromString(r.numpy())
    labels.append(ex.features.feature["label"].int64_list.value[0])

set(labels)

{0, 1, 2}

In [4]:
DATA = Path("Kaggle_Data_recheck/Cleaned_Training_Data")

counter = Counter()

for tfrec in DATA.glob("training10_*.tfrecords"):
    for r in tf.data.TFRecordDataset(str(tfrec)):
        ex = tf.train.Example.FromString(r.numpy())
        lbl = ex.features.feature["label"].int64_list.value[0]
        counter[lbl] += 1

print(counter)

2026-01-28 12:06:26.265467: I tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Counter({0: 48596, 1: 4014, 2: 3275})


In [6]:
BASE = Path("Kaggle_Data_recheck")

# Paths
val_images_path = BASE / "Incorrect_Validation_Data/cv10_data.npy"
val_labels_path = BASE / "Incorrect_Validation_Data/cv10_labels.npy"
test_images_path = BASE / "Incorrect_Test_Data/test10_data.npy"
test_labels_path = BASE / "Incorrect_Test_Data/test10_labels.npy"

# Load
val_images = np.load(val_images_path)
val_labels = np.load(val_labels_path)
test_images = np.load(test_images_path)
test_labels = np.load(test_labels_path)

# Shapes and dtypes
print("Validation images:", val_images.shape, val_images.dtype)
print("Validation labels:", val_labels.shape, val_labels.dtype)
print("Test images:", test_images.shape, test_images.dtype)
print("Test labels:", test_labels.shape, test_labels.dtype)

# Value ranges
print("Validation image range:", val_images.min(), val_images.max())
print("Test image range:", test_images.min(), test_images.max())

# Label distributions
print("Validation label distribution:", Counter(val_labels.tolist()))
print("Test label distribution:", Counter(test_labels.tolist()))

Validation images: (7682, 299, 299, 1) uint8
Validation labels: (7682,) int64
Test images: (7682, 299, 299, 1) uint8
Test labels: (7682,) int64
Validation image range: 0 255
Test image range: 0 255
Validation label distribution: Counter({0: 6680, 1: 558, 3: 369, 4: 39, 2: 36})
Test label distribution: Counter({0: 6680, 2: 606, 4: 396})


<function abs(x, /)>

In [7]:
# Load images and labels
val_images = np.load(BASE / "Incorrect_Validation_Data/cv10_data.npy")
val_labels = np.load(BASE / "Incorrect_Validation_Data/cv10_labels.npy")
test_images = np.load(BASE / "Incorrect_Test_Data/test10_data.npy")
test_labels = np.load(BASE / "Incorrect_Test_Data/test10_labels.npy")

# Concatenate
all_images = np.concatenate([val_images, test_images], axis=0)
all_labels = np.concatenate([val_labels, test_labels], axis=0)

# Remap labels: 0:0, 1:1, 2:1, 3:2, 4:2
remap = np.vectorize(lambda x: 1 if x == 2 else 2 if x >= 3 else x)
all_labels = remap(all_labels)

# Shuffle but preserve correspondence
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(all_labels))
all_images = all_images[indices]
all_labels = all_labels[indices]

# Split into new validation and test sets
val_split = len(val_labels)  # keep original validation size
new_val_images = all_images[:val_split]
new_val_labels = all_labels[:val_split]
new_test_images = all_images[val_split:]
new_test_labels = all_labels[val_split:]

# Save cleaned data
CLEANED_VAL = BASE / "Cleaned_Validation_Data"
CLEANED_TEST = BASE / "Cleaned_Test_Data"
CLEANED_VAL.mkdir(exist_ok=True)
CLEANED_TEST.mkdir(exist_ok=True)

np.save(CLEANED_VAL / "cv10_data.npy", new_val_images)
np.save(CLEANED_VAL / "cv10_labels.npy", new_val_labels)
np.save(CLEANED_TEST / "test10_data.npy", new_test_images)
np.save(CLEANED_TEST / "test10_labels.npy", new_test_labels)

print("Cleaned and shuffled Validation/Test sets saved.")

Cleaned and shuffled Validation/Test sets saved.


In [8]:
CLEANED_VAL = BASE / "Cleaned_Validation_Data"
CLEANED_TEST = BASE / "Cleaned_Test_Data"

# Load cleaned labels
val_labels = np.load(CLEANED_VAL / "cv10_labels.npy")
test_labels = np.load(CLEANED_TEST / "test10_labels.npy")

# Frequency counts
print("Validation label distribution:", Counter(val_labels.tolist()))
print("Test label distribution:", Counter(test_labels.tolist()))

Validation label distribution: Counter({0: 6663, 1: 593, 2: 426})
Test label distribution: Counter({0: 6697, 1: 607, 2: 378})
